# Notebook 09 — CP03: Parameter Estimation and Model Comparison

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 9 of 12  
**Time estimate:** 75 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy import stats

rng = np.random.default_rng(42)

# ---- Generate synthetic observed incidence data ----
N    = 1000; BETA_TRUE = 0.30; GAMMA_TRUE = 0.10
T_MAX = 60; dt_obs = 1.0
t_obs = np.arange(0, T_MAX + 1, dt_obs)

def run_sir(beta, gamma, N, I0, t_eval):
    sol = solve_ivp(lambda t, y: [-beta*y[0]*y[1]/N, beta*y[0]*y[1]/N - gamma*y[1], gamma*y[1]],
                    [0, t_eval[-1]], [N-I0, I0, 0], t_eval=t_eval)
    return sol.y

S_true, I_true, R_true = run_sir(BETA_TRUE, GAMMA_TRUE, N, 5, t_obs)
# Observed: new cases per day = dR/dt = gamma * I, with Poisson noise
new_cases_true = np.diff(R_true)
observed = rng.poisson(np.maximum(0.1, new_cases_true)).astype(float)
print(f'Observed incidence: {len(observed)} time points, total cases: {observed.sum():.0f}')

# ---- Log-likelihood function ----
def log_likelihood(beta, gamma, N, I0, observed, t_obs):
    if beta <= 0 or gamma <= 0 or beta > 2 or gamma > 1:
        return -1e10
    try:
        _, I_hat, R_hat = run_sir(beta, gamma, N, I0, t_obs)
        new_cases_hat = np.diff(R_hat)
        new_cases_hat = np.maximum(0.1, new_cases_hat)
        ll = stats.poisson.logpmf(observed.astype(int), new_cases_hat).sum()
        return ll if np.isfinite(ll) else -1e10
    except:
        return -1e10

# ---- Metropolis-Hastings MCMC ----
N_SAMPLES = 3000; BURN_IN = 1000
PROPOSAL_STD = np.array([0.02, 0.01])  # for [beta, gamma]

chain = np.zeros((N_SAMPLES, 2))
chain[0] = [0.25, 0.12]  # starting values
lp_current = log_likelihood(chain[0,0], chain[0,1], N, 5, observed, t_obs)
n_accept = 0

for i in range(1, N_SAMPLES):
    proposal = chain[i-1] + rng.normal(0, PROPOSAL_STD)
    lp_prop = log_likelihood(proposal[0], proposal[1], N, 5, observed, t_obs)
    log_ratio = lp_prop - lp_current
    if np.log(rng.random()) < log_ratio:
        chain[i] = proposal; lp_current = lp_prop; n_accept += 1
    else:
        chain[i] = chain[i-1]

chain_post = chain[BURN_IN:]
print(f'Acceptance rate: {n_accept/(N_SAMPLES-1):.1%}')
print(f'β posterior: {chain_post[:,0].mean():.3f} ± {chain_post[:,0].std():.3f} (true: {BETA_TRUE})')
print(f'γ posterior: {chain_post[:,1].mean():.3f} ± {chain_post[:,1].std():.3f} (true: {GAMMA_TRUE})')

In [ ]:
# ---- Profile likelihood ----
beta_grid = np.linspace(0.15, 0.45, 50)
gamma_grid = np.linspace(0.05, 0.20, 50)

# Profile likelihood for beta (maximized over gamma)
profile_beta = []
for b in beta_grid:
    ll_vals = [log_likelihood(b, g, N, 5, observed, t_obs) for g in gamma_grid]
    profile_beta.append(max(ll_vals))
profile_beta = np.array(profile_beta) - max(profile_beta)  # normalize to 0

# Profile likelihood for gamma
profile_gamma = []
for g in gamma_grid:
    ll_vals = [log_likelihood(b, g, N, 5, observed, t_obs) for b in beta_grid]
    profile_gamma.append(max(ll_vals))
profile_gamma = np.array(profile_gamma) - max(profile_gamma)

# 95% CI: where profile likelihood > -1.92 (chi-squared threshold / 2)
CI95_THRESHOLD = -1.92
beta_ci  = beta_grid[(profile_beta > CI95_THRESHOLD)]
gamma_ci = gamma_grid[(profile_gamma > CI95_THRESHOLD)]

# ---- AIC comparison: SIR vs exponential growth ----
def aic(n_params, max_ll): return 2*n_params - 2*max_ll

max_ll_sir = log_likelihood(chain_post[:,0].mean(), chain_post[:,1].mean(), N, 5, observed, t_obs)
# Exponential growth baseline: lambda_t = lambda_0 * exp(r*t)
t_growth = np.arange(len(observed))
from scipy.optimize import minimize_scalar
def neg_ll_exp(r):
    lam0 = observed[0] if observed[0] > 0 else 1.0
    lam = lam0 * np.exp(r * t_growth[:len(observed)])
    return -stats.poisson.logpmf(observed.astype(int), np.maximum(0.1, lam)).sum()
res = minimize_scalar(neg_ll_exp, bounds=(-0.5, 0.5), method='bounded')
max_ll_exp = -res.fun

aic_sir = aic(2, max_ll_sir)
aic_exp = aic(2, max_ll_exp)
delta_aic = aic_exp - aic_sir
print(f'AIC SIR: {aic_sir:.1f}, AIC Exponential: {aic_exp:.1f}, ΔAIC={delta_aic:.1f} (SIR preferred if ΔAIC > 0)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Trace plots
ax = axes[0, 0]
ax.plot(chain[:, 0], color='#4e79a7', lw=0.8, alpha=0.8, label='β')
ax.axhline(BETA_TRUE, color='#4e79a7', ls='--', lw=2)
ax.plot(chain[:, 1], color='#e15759', lw=0.8, alpha=0.8, label='γ')
ax.axhline(GAMMA_TRUE, color='#e15759', ls='--', lw=2)
ax.axvline(BURN_IN, color='grey', ls=':', lw=1.5, label=f'Burn-in ({BURN_IN})')
ax.set_xlabel('MCMC iteration'); ax.set_ylabel('Parameter value')
ax.set_title('A. MCMC trace plots\n(dashed = true value)')
ax.legend(fontsize=8)
ax.text(-0.12, 1.05, 'A', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel B: Posterior distributions
ax = axes[0, 1]
ax.hist(chain_post[:, 0], bins=30, color='#4e79a7', alpha=0.7, density=True, label=f'β (true={BETA_TRUE})')
ax.hist(chain_post[:, 1], bins=30, color='#e15759', alpha=0.7, density=True, label=f'γ (true={GAMMA_TRUE})')
ax.axvline(BETA_TRUE, color='#4e79a7', ls='--', lw=2)
ax.axvline(GAMMA_TRUE, color='#e15759', ls='--', lw=2)
ax.set_xlabel('Parameter value'); ax.set_ylabel('Posterior density')
ax.set_title('B. Posterior distributions\n(dashed = true value; post burn-in samples)')
ax.legend(fontsize=8)
ax.text(-0.12, 1.05, 'B', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel C: Profile likelihood
ax = axes[1, 0]
ax.plot(beta_grid, profile_beta, color='#4e79a7', lw=2, label='β profile')
ax.axhline(CI95_THRESHOLD, color='grey', ls='--', lw=1, label='95% CI threshold (-1.92)')
if len(beta_ci) > 0:
    ax.fill_between(beta_ci, CI95_THRESHOLD, 0, alpha=0.3, color='#4e79a7')
    ax.axvline(beta_ci[0], color='#4e79a7', ls=':', lw=1)
    ax.axvline(beta_ci[-1], color='#4e79a7', ls=':', lw=1)
ax.axvline(BETA_TRUE, color='black', ls='-', lw=2, label=f'True β={BETA_TRUE}')
ax.set_xlabel('β'); ax.set_ylabel('Profile log-likelihood (normalized)')
ax.set_title(f'C. Profile likelihood for β\n(95% CI: [{beta_ci[0]:.2f}, {beta_ci[-1]:.2f}] if coverage exists)')
ax.legend(fontsize=8)
ax.text(-0.12, 1.05, 'C', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel D: Observed vs fitted
ax = axes[1, 1]
beta_fit  = chain_post[:,0].mean()
gamma_fit = chain_post[:,1].mean()
_, I_fit, R_fit = run_sir(beta_fit, gamma_fit, N, 5, t_obs)
new_cases_fit = np.diff(R_fit)
ax.bar(t_obs[1:], observed, color='grey', alpha=0.5, label='Observed (Poisson noise)')
ax.plot(t_obs[1:], new_cases_fit, color='#e15759', lw=2.5, label=f'Fitted (β={beta_fit:.2f}, γ={gamma_fit:.2f})')
ax.plot(t_obs[1:], np.diff(R_true), color='black', ls='--', lw=1.5, label='True (no noise)')
ax.set_xlabel('Day'); ax.set_ylabel('New cases per day')
ax.set_title(f'D. Observed vs fitted incidence\n(AIC SIR={aic_sir:.0f} vs Exp={aic_exp:.0f}, ΔAIC={delta_aic:.0f})')
ax.legend(fontsize=8)
ax.text(-0.12, 1.05, 'D', transform=ax.transAxes, fontsize=12, fontweight='bold')

plt.suptitle('Module 20 CP03: Parameter Estimation and Model Comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cp03_parameter_estimation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — Reflection

> *[What does ΔAIC > 10 between SIR and exponential growth tell you about
> which model is preferred? What is the biological interpretation: why does
> exponential growth fail to fit epidemic data after the peak?]*

---
*Next: `10_portfolio_construction.ipynb`*